<a href="https://colab.research.google.com/github/ArtIC-TITECH/b3-proj-2026/blob/main/theme_F/theme_F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# テーマF：Looped LLMにおける重み量子化ビット幅と精度・モデルサイズのトレードオフ

## 目的

Looped LLM **Ouro-1.4B** にweight-only post-training quantization（PTQ）を適用し、重みビット幅を下げたときの性能低下と理論モデルサイズの削減量を比較する。

- **独立変数**：重みビット幅
- **統制変数**：再帰回数（4回）、評価データ、系列長、量子化対象、復号方法
- **評価指標**：Perplexity、next-token accuracy、理論モデルサイズ、生成例

ここで行うのは低bit整数演算による高速化ではなく、量子化誤差を評価する**fake quantization**である。実行時間ではなく、ビット幅と品質・保存量の関係を研究対象とする。


## 環境セットアップ

Colabで「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選ぶ。次のセルでパッケージが更新された場合だけ、セッションを再起動して先頭から実行する。


In [ ]:
# Ouroの固定revisionに合わせ、Transformers周辺だけ互換版へ揃えます。
# torch / CUDAはColab標準環境をそのまま使います。
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
import torch

if not torch.cuda.is_available():
    raise RuntimeError("ランタイムのタイプをGPU（T4）に変更してください。")

TARGETS = {
    "transformers": "4.55.0",
    "tokenizers": "0.21.4",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.10.0",
    "safetensors": "0.6.2",
    "Pillow": "11.3.0",
}

needs_install = []
for pkg, wanted in TARGETS.items():
    try:
        current = version(pkg)
    except PackageNotFoundError:
        current = None
    if current != wanted:
        needs_install.append(f"{pkg}=={wanted}")

RESTART_REQUIRED = bool(needs_install)
if needs_install:
    print("互換パッケージをインストールします:", needs_install)
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-cache-dir", "--upgrade", "--force-reinstall",
            "--no-deps", "--only-binary=:all:", *needs_install,
        ],
        check=True,
    )
    print("\nインストール完了。『ランタイム』→『セッションを再起動』を1回実行してください。")
    print("再起動後は、このNotebookを先頭から実行してください。")
else:
    print("環境OK:", torch.__version__, torch.cuda.get_device_name(0))


## モジュールの読み込み・GPU確認


In [ ]:
import gc
import math
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw

if globals().get("RESTART_REQUIRED", False):
    raise RuntimeError("パッケージ更新後です。ランタイムを再起動して先頭から実行してください。")

required_versions = {
    "transformers": "4.55.0",
    "tokenizers": "0.21.4",
    "huggingface-hub": "0.36.2",
    "accelerate": "1.10.0",
    "safetensors": "0.6.2",
}
for pkg, wanted in required_versions.items():
    try:
        current = version(pkg)
    except PackageNotFoundError:
        current = None
    if current != wanted:
        raise RuntimeError(f"{pkg}={current!r} です。環境セットアップを再実行してください。")

ImageDraw.Draw(Image.new("RGB", (8, 8))).point((0, 0))

from datasets import load_dataset
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

device = "cuda:0"
capability = torch.cuda.get_device_capability(0)
dtype = torch.bfloat16 if capability[0] >= 8 and torch.cuda.is_bf16_supported() else torch.float16
BASELINE_LABEL = "BF16" if dtype == torch.bfloat16 else "FP16"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "Transformers:", version("transformers"))
print("Baseline dtype:", BASELINE_LABEL)


## 実験条件

まず結果を予想する。どのビット幅までは精度を保てるか、どこで急激に劣化するかを記録してから実験すること。デフォルト設定は動作確認用であり、最終結果では評価系列数を増やして再現性を確認する。


In [ ]:
MODEL_ID = "ByteDance/Ouro-1.4B"
MODEL_REVISION = "574fa66cb8bf5abdc979642d01cf2b79b16bfab1"

# 独立変数：重みビット幅。16bitを基準条件として必ず含めます。
BITS_LIST = [16, 8, 6, 4, 3, 2]

# 統制変数：再帰回数・評価データ・系列長・復号方法
NUM_RECURRENT_STEPS = 4
SEQ_LEN = 256
NUM_SEQUENCES = 16       # 最終評価では32または64への増加を推奨
EVAL_BATCH_SIZE = 1      # T4向け

print("Bit-widths:", BITS_LIST)
print("Fixed recurrent steps:", NUM_RECURRENT_STEPS)
print("Evaluation predictions:", SEQ_LEN * NUM_SEQUENCES)


## Ouro-1.4Bの読み込み


In [ ]:
for name in ("model", "target_modules", "original_weights"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

config = AutoConfig.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
)
config.total_ut_steps = NUM_RECURRENT_STEPS
config.use_cache = False
config.early_exit_threshold = None
config.early_exit_step = NUM_RECURRENT_STEPS - 1

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    config=config,
    trust_remote_code=True,
    torch_dtype=dtype,
    attn_implementation="eager",
    low_cpu_mem_usage=True,
).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e9:.3f} B")
print("Recurrent steps:", model.config.total_ut_steps)

with torch.inference_mode():
    test_ids = tokenizer("The sky is blue because", return_tensors="pt").input_ids.to(device)
    test_out = model(input_ids=test_ids, use_cache=False, logits_to_keep=1)
    assert torch.isfinite(test_out.logits).all(), "試験出力にNaN/Infがあります"
    print("CUDA forward: OK", tuple(test_out.logits.shape))
    del test_ids, test_out


## 評価データ：WikiText-2


In [ ]:
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="test",
)
texts = [text for text in dataset["text"] if text.strip()]
all_text = "\n\n".join(texts)

tokens_per_sequence = SEQ_LEN + 1
needed_tokens = NUM_SEQUENCES * tokens_per_sequence
token_ids = tokenizer(
    all_text,
    return_tensors="pt",
    add_special_tokens=False,
    truncation=True,
    max_length=needed_tokens,
).input_ids[0]

if token_ids.numel() < needed_tokens:
    raise RuntimeError(f"評価用tokenが不足しています: {token_ids.numel()} < {needed_tokens}")

eval_tokens = token_ids[:needed_tokens].view(NUM_SEQUENCES, tokens_per_sequence)
print("eval_tokens:", tuple(eval_tokens.shape))


## Per-channel対称一様量子化

`nn.Linear` の各出力チャネルについて絶対値最大値からscaleを求め、round-to-nearestで整数格子へ写した後に浮動小数点へ戻す。各ビット幅は必ず同じ16bit基準重みから生成し、条件間で量子化誤差を累積させない。


In [ ]:
@torch.inference_mode()
def evaluate_language_model(model, eval_tokens, batch_size=1, device="cuda:0"):
    """Perplexityとnext-token accuracyを同じtoken集合で測る。"""
    model.eval()
    total_nll, total_correct, total_tokens = 0.0, 0, 0

    for start in range(0, len(eval_tokens), batch_size):
        batch = eval_tokens[start:start + batch_size].to(device)
        labels = batch[:, 1:]
        outputs = model(input_ids=batch[:, :-1], use_cache=False, logits_to_keep=0)
        logits = outputs.logits

        if not torch.isfinite(logits).all():
            return {
                "perplexity": float("inf"),
                "token_accuracy_pct": float("nan"),
                "status": "non-finite logits",
            }

        # 語彙方向が大きいため、位置方向を小分けにしてFP32で集計する。
        for pos in range(0, labels.shape[1], 32):
            scores = logits[:, pos:pos + 32, :].float()
            target = labels[:, pos:pos + 32]
            loss = F.cross_entropy(
                scores.reshape(-1, scores.shape[-1]),
                target.reshape(-1),
                reduction="sum",
            )
            total_nll += loss.item()
            total_correct += (scores.argmax(dim=-1) == target).sum().item()
            total_tokens += target.numel()

        del batch, labels, outputs, logits

    mean_nll = total_nll / total_tokens
    return {
        "perplexity": math.exp(mean_nll) if mean_nll < 709 else float("inf"),
        "token_accuracy_pct": 100.0 * total_correct / total_tokens,
        "status": "ok",
    }


SKIP_MODULE_NAMES = ("lm_head", "early_exit_gate")


def get_target_linear_modules(model):
    """埋め込み・出力head・exit gateを除き、Linear重みを量子化対象にする。"""
    return {
        name: module
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear)
        and not any(skip in name for skip in SKIP_MODULE_NAMES)
    }


@torch.no_grad()
def symmetric_per_channel_fake_quantize(weight, num_bits):
    """出力チャネルごとの対称一様fake quantization（round-to-nearest）。"""
    if num_bits >= 16:
        return weight

    qmax = 2 ** (num_bits - 1) - 1
    w = weight.float()
    max_abs = w.abs().amax(dim=1, keepdim=True)
    scale = (max_abs / qmax).clamp_min(1e-8)
    q = torch.clamp(torch.round(w / scale), -qmax, qmax)
    return (q * scale).to(weight.dtype)


@torch.no_grad()
def apply_weight_quantization(model, original_weights, num_bits):
    modules = dict(model.named_modules())
    for name, original_cpu in original_weights.items():
        module = modules[name]
        original = original_cpu.to(module.weight.device, module.weight.dtype)
        quantized = original if num_bits >= 16 else symmetric_per_channel_fake_quantize(
            original, num_bits
        )
        module.weight.copy_(quantized)
        del original, quantized
    torch.cuda.empty_cache()


def estimate_model_size_mb(model, n_quant_params, n_scale_values, num_bits):
    """対象重みだけを低bit保存した場合の理論サイズ。実ファイルサイズではない。"""
    total_params = sum(p.numel() for p in model.parameters())
    unquantized_params = total_params - n_quant_params
    quant_bits = 16 if num_bits >= 16 else num_bits
    weight_bytes = n_quant_params * quant_bits / 8
    unquantized_bytes = unquantized_params * 16 / 8
    scale_bytes = 0 if num_bits >= 16 else n_scale_values * 16 / 8
    return (weight_bytes + unquantized_bytes + scale_bytes) / 1024**2


In [ ]:
target_modules = get_target_linear_modules(model)
n_quant_params = sum(module.weight.numel() for module in target_modules.values())
n_scale_values = sum(module.weight.shape[0] for module in target_modules.values())

# 各条件を同じ基準重みから始めるため、元重みをCPUに1回だけ保存する。
original_weights = {
    name: module.weight.detach().cpu().clone()
    for name, module in target_modules.items()
}

print("Quantized Linear modules:", len(target_modules))
print(f"Quantized parameters: {n_quant_params / 1e9:.3f} B")
print(f"Fraction of all parameters: {100 * n_quant_params / n_params:.1f}%")


## ビット幅を変えた評価


In [ ]:
results = []

for num_bits in BITS_LIST:
    label = BASELINE_LABEL if num_bits >= 16 else f"INT{num_bits}"
    print("\n" + "=" * 60)
    print("Condition:", label)

    apply_weight_quantization(model, original_weights, num_bits)
    metrics = evaluate_language_model(
        model, eval_tokens, batch_size=EVAL_BATCH_SIZE, device=device
    )
    size_mb = estimate_model_size_mb(
        model, n_quant_params, n_scale_values, num_bits
    )
    results.append({
        "bits": num_bits,
        "label": label,
        "status": metrics["status"],
        "perplexity": metrics["perplexity"],
        "token_accuracy_pct": metrics["token_accuracy_pct"],
        "theoretical_size_mb": size_mb,
    })
    print(results[-1])

results_df = pd.DataFrame(results)
baseline = results_df.loc[results_df["bits"] == 16].iloc[0]
results_df["ppl_ratio_vs_16bit"] = results_df["perplexity"] / baseline["perplexity"]
results_df["accuracy_drop_pp"] = (
    baseline["token_accuracy_pct"] - results_df["token_accuracy_pct"]
)
results_df.to_csv("theme_F_quantization_results.csv", index=False)
results_df.round(4)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(results_df["bits"], results_df["perplexity"], marker="o")
axes[0].set_xlabel("Weight bit-width")
axes[0].set_ylabel("Perplexity (lower is better)")
axes[0].invert_xaxis()

axes[1].plot(results_df["bits"], results_df["token_accuracy_pct"], marker="o")
axes[1].set_xlabel("Weight bit-width")
axes[1].set_ylabel("Next-token accuracy (%)")
axes[1].invert_xaxis()

axes[2].plot(results_df["theoretical_size_mb"], results_df["perplexity"], marker="o")
for _, row in results_df.iterrows():
    axes[2].annotate(row["label"], (row["theoretical_size_mb"], row["perplexity"]))
axes[2].set_xlabel("Theoretical model size (MB)")
axes[2].set_ylabel("Perplexity")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 生成文の比較

自動指標だけでは捉えにくい破綻を確認する。事実説明と簡単な推論など、性質の異なるプロンプトを使い、全条件で同じgreedy decodingを適用する。


In [ ]:
@torch.inference_mode()
def generate_response(model, tokenizer, prompt, max_new_tokens=48):
    encoded = tokenizer(prompt, return_tensors="pt")
    ids = encoded.input_ids.to(device)
    mask = encoded.attention_mask.to(device)
    start = ids.shape[1]
    eos = tokenizer.eos_token_id

    for _ in range(max_new_tokens):
        out = model(
            input_ids=ids,
            attention_mask=mask,
            use_cache=False,
            logits_to_keep=1,
        )
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        token = next_id.item()
        ids = torch.cat([ids, next_id], dim=1)
        mask = torch.cat([mask, torch.ones_like(next_id)], dim=1)
        if eos is not None and token == eos:
            break
    return tokenizer.decode(ids[0, start:], skip_special_tokens=True)


# TODO: 性質の異なるプロンプトを最低2つにする。
PROMPTS = [
    "Explain why the sky looks blue in three sentences.",
]
GENERATION_BITS = [16, 4, 2]

generation_rows = []
for num_bits in GENERATION_BITS:
    apply_weight_quantization(model, original_weights, num_bits)
    label = BASELINE_LABEL if num_bits >= 16 else f"INT{num_bits}"
    for prompt in PROMPTS:
        response = generate_response(model, tokenizer, prompt)
        generation_rows.append({"label": label, "prompt": prompt, "response": response})
        print(f"\n[{label}] {prompt}\n{response}")

generation_df = pd.DataFrame(generation_rows)
generation_df.to_csv("theme_F_generation_results.csv", index=False)

# 後続実験に影響を残さないよう16bit基準重みに戻す。
apply_weight_quantization(model, original_weights, 16)


## 課題

- 16/8/6/4/3/2-bitについて、Perplexity、next-token accuracy、理論モデルサイズを表とグラフで比較する。
- 「ほとんど性能を落とさず圧縮できる範囲」と「性能が急激に崩れる境界」を定め、その根拠を述べる。
- 生成文を最低2種類のプロンプトで比較し、自動指標の変化と目視評価が一致するか考察する。
- 評価系列数を増やす、量子化対象層を限定する、per-tensor量子化と比較する、のいずれか1つ以上を追加実験する。
- fake quantizationの結果だけから実機上の高速化を主張できない理由を説明する。

**提出物**：実験条件、予想、結果表・グラフ、追加実験、考察、結論をまとめる。最良条件だけでなく、失敗した条件も結果として報告すること。
